### LangChain Expression Language (LCEL)

LangChain Expression Language (LCEL) allows building chains step-by-step using modular Runnable primitives.

---

### LCEL Primitives Quick Reference

| Concept / Primitive | Short Description |
| :--- | :--- |
| Pipe Operator (|) | Connects components sequentially. |
| invoke() | Executes chain synchronously on 1 input. |
| stream() | Streams text chunks token-by-token. |
| batch() | Runs chain concurrently on a list of inputs. |
| RunnableSequence | Explicit sequential chain object. |
| RunnableLambda | Wraps a custom Python function. |
| RunnableParallel | Runs multiple sub-chains at the same time. |
| RunnablePassthrough | Passes input through unchanged. |
| assign() | Adds new key-value pairs to input dictionary. |
| RunnableBranch | Routes input based on if/else conditions. |
| with_config() | Sets runtime tags and run names. |
| with_retry() | Retries execution automatically on error. |
| with_fallbacks() | Switches to a backup chain if primary fails. |

### 1. Environment Setup

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gemini-3.6-flash", model_provider="google_genai")
prompt = ChatPromptTemplate.from_template("Explain {topic} in 1 sentence.")
parser = StrOutputParser()

### 2. The Pipe Operator (|)

The pipe operator (|) connects prompt, model, and output parser sequentially into a single runnable chain.

In [ ]:
# Create chain using pipe operator |
chain = prompt | model | parser

### 3. invoke() Method

The invoke() method executes a chain synchronously on a single input dictionary and returns the final parsed result.

In [ ]:
result = chain.invoke({"topic": "Python"})
print("Result:\n", result)

### 4. stream() Method

The stream() method yields response text chunks token-by-token as they are generated by the model.

In [ ]:
for chunk in chain.stream({"topic": "Solar System"}):
    print(chunk, end="|", flush=True)
print()

### 5. batch() Method

The batch() method processes a list of multiple input dictionaries concurrently in parallel.

In [ ]:
inputs = [{"topic": "Docker"}, {"topic": "Kubernetes"}]
results = chain.batch(inputs)

for topic_dict, res in zip(inputs, results):
    print(f"{topic_dict['topic']} -> {res}")

### 6. RunnableSequence

RunnableSequence is the explicit class underlying the pipe operator. You can instantiate it directly with a list of runnables.

In [ ]:
from langchain_core.runnables import RunnableSequence

explicit_chain = RunnableSequence(prompt, model, parser)
print(explicit_chain.invoke({"topic": "Linux"}))

### 7. RunnableLambda

RunnableLambda wraps a custom Python function into an LCEL runnable component so it can be used inside chains.

In [ ]:
from langchain_core.runnables import RunnableLambda

def uppercase_text(text: str) -> str:
    return text.upper()

uppercase_runnable = RunnableLambda(uppercase_text)

# Chain containing custom python function
custom_chain = chain | uppercase_runnable
print(custom_chain.invoke({"topic": "Git"}))

### 8. RunnableParallel

RunnableParallel executes multiple sub-chains simultaneously on the exact same input dictionary.

In [ ]:
from langchain_core.runnables import RunnableParallel

fact_prompt = ChatPromptTemplate.from_template("Give 1 fun fact about {topic}.")
joke_prompt = ChatPromptTemplate.from_template("Tell 1 joke about {topic}.")

fact_chain = fact_prompt | model | parser
joke_chain = joke_prompt | model | parser

parallel_chain = RunnableParallel(
    fact=fact_chain,
    joke=joke_chain
)

output = parallel_chain.invoke({"topic": "Robots"})
print("Fact:", output["fact"])
print("Joke:", output["joke"])

### 9. RunnablePassthrough

RunnablePassthrough passes input dictionary data through unchanged down the pipeline without modification.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

passthrough_chain = RunnablePassthrough() | chain
print(passthrough_chain.invoke({"topic": "Database"}))

### 10. RunnablePassthrough.assign()

The assign() method adds new computed key-value pairs to the input dictionary while preserving all existing keys.

In [ ]:
# Add a extra_info key dynamically to input dictionary
assign_chain = RunnablePassthrough.assign(
    extra_info=lambda x: f"Topic in uppercase: {x['topic'].upper()}"
)

state = assign_chain.invoke({"topic": "AI"})
print("Updated State Dictionary:", state)

### 11. RunnableBranch (Conditional Routing)

RunnableBranch routes inputs to different sub-chains based on boolean condition functions.

In [ ]:
from langchain_core.runnables import RunnableBranch

math_prompt = ChatPromptTemplate.from_template("Solve math: {question}") | model | parser
text_prompt = ChatPromptTemplate.from_template("Answer question: {question}") | model | parser

def is_math(x: dict) -> bool:
    return "plus" in x["question"] or "+" in x["question"]

branch = RunnableBranch(
    (is_math, math_prompt),
    text_prompt
)

print("Math Question:", branch.invoke({"question": "What is 5 plus 5?"}))
print("Text Question:", branch.invoke({"question": "What is the capital of Japan?"}))

### 12. with_config()

The with_config() method attaches metadata such as run names or tags to a runnable chain for tracking and monitoring.

In [ ]:
configured_chain = chain.with_config(
    run_name="SimpleTopicExplainer",
    tags=["demo", "tutorial"]
)

print(configured_chain.invoke({"topic": "API"}))

### 13. with_retry()

The with_retry() method configures automatic retry attempts if execution encounters an exception or error.

In [ ]:
retrying_chain = chain.with_retry(
    stop_after_attempt=3
)

print(retrying_chain.invoke({"topic": "Encryption"}))

### 14. with_fallbacks()

The with_fallbacks() method attaches backup runnables that run automatically if the primary runnable fails.

In [ ]:
backup_model = init_chat_model("gemini-3.6-flash", model_provider="google_genai", temperature=0.0)
backup_chain = prompt | backup_model | parser

resilient_chain = chain.with_fallbacks([backup_chain])

print(resilient_chain.invoke({"topic": "Microprocessors"}))